<a href="https://colab.research.google.com/github/amandarcrangel/project_studies_fiap/blob/main/Amostragem.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Amostragem

In [ ]:
import numpy as np
import pandas as pd
import random as rd #números aleatorios
from sklearn.model_selection import train_test_split #train_test_split usada para fazer amostragem estratificada - machine learning

Dados

In [ ]:
dados = pd.read_csv('dados_renda_municipios.csv',
                    sep=",",
                    decimal=".",
                    encoding="utf-8")

In [ ]:
dados.head()

,UF,Municipio,RDPC
0,Rondônia,ALTA FLORESTA D'OESTE,476.99
1,Rondônia,ARIQUEMES,689.95
2,Rondônia,CABIXI,457.17
3,Rondônia,CACOAL,738.06
4,Rondônia,CEREJEIRAS,577.18


In [ ]:
dados.shape #quantos dados tempos

(5565, 3)

## Amostragem por estado

- selecionar o estado

In [ ]:
 #ajustes iniciais
 uf= "São Paulo"

In [ ]:
dados_municipio=dados [dados["UF"] == uf].reset_index(drop=True) #== filtrar o UF que é municipio

In [ ]:
dados_municipio.shape

(645, 3)

In [ ]:
#criacao dos estratos
dados_municipio["classe_renda"] = pd.qcut(dados_municipio["RDPC"], 4, labels=["D", "C", "B", "A"]) #QCUT QUARTIS dividimos por quatro grupos

In [ ]:
dados_municipio.head()

,UF,Municipio,RDPC,classe_renda
0,São Paulo,ADAMANTINA,975.43,A
1,São Paulo,ADOLFO,661.65,C
2,São Paulo,AGUAÍ,636.07,C
3,São Paulo,ÁGUAS DA PRATA,853.39,A
4,São Paulo,ÁGUAS DE LINDÓIA,730.13,B


- Analises piloto

In [ ]:
#dados gerais amostragem piloto
dados_piloto = dados_municipio.agg(media_RDPC = pd.NamedAgg(column="RDPC", aggfunc="mean"),
                                   dp_RDPC = pd.NamedAgg(column="RDPC", aggfunc="std"),
                                   n = pd.NamedAgg(column="RDPC", aggfunc="count"))

In [ ]:
dados_piloto

,RDPC
media_RDPC,713.926155
dp_RDPC,197.398270
n,645.000000


In [ ]:
#dados piloto por estrato
dados_piloto_classe = dados_municipio.groupby("classe_renda") \
                                   .agg(media_RDPC = pd.NamedAgg(column="RDPC", aggfunc="mean"), \
                                         dp_RDPC = pd.NamedAgg(column="RDPC", aggfunc="std"), \
                                         n = pd.NamedAgg(column="RDPC", aggfunc="count")) \
                                   .reset_index()

<ipython-input-18-6232625a4117>:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  dados_piloto_classe = dados_municipio.groupby("classe_renda") \


In [ ]:
dados_piloto_classe

,classe_renda,media_RDPC,dp_RDPC,n
0,D,517.531296,58.889964,162
1,C,637.659259,28.839032,162
2,B,736.007313,34.142763,160
3,A,966.337453,204.484141,161


##Tamanho da amostra

In [ ]:
# formula discreta crinado a formala matématica
def formula_amostra_discreta(N, Z, ME):
  N = (Z**2 * 0.25 * N) / ((ME**2) + (Z**2 * 0.25) * (N-1) / N)
  return int(n)

In [ ]:
# formula continua
def formula_amostra_continua(N, S, Z, ME):
  n= (Z**2 * S**2 * N) / ((ME**2) * (N-1) + (Z**2 * S**2))
  return int(n)

- USO DA FORMULA

In [ ]:
#PARAMETROS
N= 645
Z= 1.65
S= 197.40
ME= 25

In [ ]:
#tamanho amostra
n= formula_amostra_continua(N, S, Z, ME)
n

134

- AMOSTRA ALEATÓRIA SIMPLES

- funcao random

In [ ]:
#sorteio das linhas
linhas_sorteadas = rd.sample(range(1, N+1), n)

In [ ]:
len(linhas_sorteadas)

134

In [ ]:
#filtrar os dados
dados_amostra = dados_municipio[dados_municipio.index.isin(linhas_sorteadas)]

In [ ]:
dados_amostra

,UF,Municipio,RDPC,classe_renda
1,São Paulo,ADOLFO,661.65,C
9,São Paulo,ALFREDO MARCONDES,558.38,D
13,São Paulo,ALUMÍNIO,747.27,B
17,São Paulo,ALVINLÂNDIA,710.49,B
18,São Paulo,AMERICANA,1161.68,A
...,...,...,...,...
635,São Paulo,VERA CRUZ,713.66,B
636,São Paulo,VINHEDO,1493.32,A
638,São Paulo,VISTA ALEGRE DO ALTO,732.14,B
641,São Paulo,VOTUPORANGA,977.39,A


Função Sample

In [ ]:
dados_amostra_simples = dados_municipio.sample(n=n)
dados_amostra_simples.shape

(134, 4)

In [ ]:
dados_amostra_simples.head()

,UF,Municipio,RDPC,classe_renda
527,São Paulo,SANTANA DA PONTE PENSA,856.78,A
194,São Paulo,GETULINA,608.58,C
79,São Paulo,BOM JESUS DOS PERDÕES,688.02,B
72,São Paulo,BILAC,816.06,A
223,São Paulo,IBIRAREMA,593.70,C


Amostragem Estratificada

In [ ]:
dados_amostra_estrat = train_test_split(dados_municipio,
                                        test_size= n,
                                        random_state=1245, #pesquisa que outras pessoas que vão mostar para outras pessoas, sempre vai ser o mesmo escolhido
                                        stratify=dados_municipio["classe_renda"]) [1]

In [ ]:
dados_amostra_estrat

,UF,Municipio,RDPC,classe_renda
595,São Paulo,TAPIRAÍ,449.75,D
7,São Paulo,AGUDOS,642.59,C
570,São Paulo,SARAPUÍ,619.86,C
35,São Paulo,ARAPEÍ,414.19,D
416,São Paulo,PEDREGULHO,690.08,B
...,...,...,...,...
270,São Paulo,ITIRAPUÃ,526.62,D
327,São Paulo,MARINÓPOLIS,553.24,D
152,São Paulo,DIADEMA,694.55,B
545,São Paulo,SÃO CAETANO DO SUL,2043.74,A


Avaliação

In [ ]:
dados_piloto

,RDPC
media_RDPC,713.926155
dp_RDPC,197.398270
n,645.000000


In [ ]:
dados_piloto_classe

,classe_renda,media_RDPC,dp_RDPC,n
0,D,517.531296,58.889964,162
1,C,637.659259,28.839032,162
2,B,736.007313,34.142763,160
3,A,966.337453,204.484141,161


Amostra aleatoria simples

In [ ]:
dados_amostra_simples.agg(media_RDPC = pd.NamedAgg("RDPC","mean"),
                                   dp_RDPC = pd.NamedAgg("RDPC", "std"),
                                   N = pd.NamedAgg("RDPC", "count"))

,RDPC
media_RDPC,760.734254
dp_RDPC,219.356884
N,134.000000


In [ ]:
#resumo amostra estratificada
resumo_estrat = dados_amostra_estrat.groupby("classe_renda") \
                                    .agg(media_RDPC = pd.NamedAgg("RDPC", "mean"), \
                                         dp_RDPC = pd.NamedAgg("RDPC","std"),\
                                         N = pd.NamedAgg("RDPC", "count"))\
                                    .reset_index()

<ipython-input-47-1c84cc040d8c>:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  resumo_estrat = dados_amostra_estrat.groupby("classe_renda") \


In [ ]:
resumo_estrat

,classe_renda,media_RDPC,dp_RDPC,N
0,D,526.942059,42.699055,34
1,C,633.964412,28.099760,34
2,B,735.434242,36.082491,33
3,A,982.247576,251.439577,33


In [ ]:
dados_piloto_classe

,classe_renda,media_RDPC,dp_RDPC,n
0,D,517.531296,58.889964,162
1,C,637.659259,28.839032,162
2,B,736.007313,34.142763,160
3,A,966.337453,204.484141,161
